In [1]:
import pandas as pd
import numpy as np
import os
import collections
import pysam
from Bio.Seq import Seq
from Bio import pairwise2
from Bio.pairwise2 import format_alignment
from scipy.spatial import distance
from tqdm import tqdm

/Users/loftum/miniforge3/envs/myNewEnv/lib/python3.13/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [ ]:
#NA20509_chrY	39638	18808040	18847678	yellow1	sense	NO_NGAP_PRESENT	SEQ._yellow1+_P5-AZFb-_P5-AZFb+	{'yellow1:18813580-18959702'}	Good

In [ ]:
#chrY    18813580        18959702        yellow1 92.5051 +       18813580        18959702        255,153,0

In [4]:
assemblyDict={}
directory='/LeeLab/Assemblies/HPRC_Release2/chrY_assemblies/'
for file in os.listdir(directory):
    if '.fai' in file or '.gzi' in file:
        continue
    else:
        assemblyDict[file.split("_")[0]]= directory+file

In [5]:
assemblyDict['HG01109']

'/LeeLab/Assemblies/HPRC_Release2/chrY_assemblies/HG01109_chrY.fa.gz'

In [6]:
sampleColorCoordinates={}
colorDirectory = '/LeeLab/HPRC/chromosomeY/Data/ColorBlockDataframes/lessInfo/'
for file in os.listdir(colorDirectory):
    if '.csv' in file:
        sample = file.split("_")[0]
        df = pd.read_csv(colorDirectory+file)
        if sample == 'HG03456':
            contig = str(file.split("_")[0])+"_chrY_"+str(file.split("_")[3])
        else:
            contig = file.split("_")[0]+"_chrY"
        sampleColorCoordinates[contig]=[]
        for row in df.index:
            sampleColorCoordinates[contig].append([df.at[row,'StartCoordinate'] , df.at[row,'EndCoordinate']])

In [7]:
#144 is what I should see - at the time
len(sampleColorCoordinates)

144

In [23]:
repeatMaskerDirectory = '/LeeLab/HPRC/chromosomeY/Data/RepeatMasker/'
for file in os.listdir(repeatMaskerDirectory):
    
    if '.tsv' in str(file):
        sampleName = file.split(".")[0]
        
        if sampleName in assemblyDict:
            tempDF = pd.read_csv(repeatMaskerDirectory+file,sep='\t')
            tempDF['query_match_start'] = tempDF['query_match_start'].astype(int)
            tempDF['query_match_end'] = tempDF['query_match_end'].astype(int)
            tempDF['pct_sub'] = tempDF['pct_sub'].astype(float)

    
            # THIS IS IF I WANT TO CALL IN COLORBLOCKS - Changing plans for all genome and then filter
            #Build a mask for the dataframe
            #masklist=[]
            #if sampleName == 'HG03456':
            #    coordinateList = sampleColorCoordinates[sampleName+"_chrY_1"]
            #    for block in coordinateList:
            #        masklist.append([sampleName+"_chrY_1", block[0], block[1]])
    
            #    coordinateList = sampleColorCoordinates[sampleName+"_chrY_2"]
            #    for block in coordinateList:
            #        masklist.append([sampleName+"_chrY_2", block[0], block[1]])
            #else:
    
            #    coordinateList = sampleColorCoordinates[sampleName+"_chrY"]
            #    for block in coordinateList:
            #        masklist.append([sampleName+"_chrY", block[0], block[1]])
    
            #mask = pd.Series(False, index=tempDF.index)
            #for contigName, coord_start, coord_end in masklist:
            #    mask |= (tempDF[4] == contigName) & (tempDF[5] >= coord_start) & (tempDF[6] <= coord_end)
                
            #tempDF2 = tempDF[mask].copy()
            
            Elements = ['LINE/L1','SINE/Alu','Retroposon/SVA']
            tempDF3=tempDF[(tempDF['repeat_class_family'].isin(Elements)) & (tempDF['pct_sub']<=30.0)].copy()
    
            idDict = collections.Counter(tempDF3['repeat_match_id'])
            #break\
            
            #with open('/LeeLab/HPRC/chromosomeY/Data/MEIs/part1_RM_Element_Sequences/'+sampleName+'_MEIs.fasta','a+') as outFile:
                for uniqueID in idDict.keys():
                    uniqueDF = tempDF3[tempDF3['repeat_match_id']==uniqueID].copy()
                    if min(uniqueDF['query_match_start']) >30:
                        coordinate = str([x for x in uniqueDF['query_name']][0])+":"+str(min(uniqueDF['query_match_start'])-30)+"-"+str(max(uniqueDF['query_match_end'])+30)
                        assemblies=assemblyDict[sampleName]
                        outFile.write(pysam.faidx(assemblies, coordinate))
                    else:
                        coordinate = str([x for x in uniqueDF['query_name']][0])+":"+str(min(uniqueDF['query_match_start'])-(min(uniqueDF['query_match_start'])-1))+"-"+str(max(uniqueDF['query_match_end'])+30)
                        assemblies=assemblyDict[sampleName]
                        outFile.write(pysam.faidx(assemblies, coordinate))
            outFile.close()
      
    else:
        continue
        

## RepeatMask the elements

## Read in Limeaid Output

In [1]:
from Bio.Seq import Seq
import os

In [3]:
goodElements=['Retroposon/SVA','SINE/Alu','LINE/L1']
ElementDict={x:{} for x in goodElements}

In [4]:
directory="/LeeLab/HPRC/chromosomeY/Data/MEIs/part2_limeaid/limeaid_MEI_out"
for file in os.listdir(directory):
    if '.tsv' in file:
        limeaid = pd.read_csv(directory+"/"+file,sep='\t')
        goodElements=['Retroposon/SVA','SINE/Alu','LINE/L1']
        nonElementDF = limeaid[~limeaid['Element_Designation'].isin(goodElements)].copy()
        ElementDF = limeaid[limeaid['Element_Designation'].isin(goodElements)].copy()
        decentINS = ElementDF[(ElementDF['FLAGS']=='No_Flags') & (ElementDF['Tail_Type']!='No_Tail_Type')].copy()

        for element in goodElements:
            tempDF = decentINS[decentINS['Element_Designation']==element].copy()
            for row in tempDF.index:  
                name = str(tempDF.at[row,'ID'])+"-"+str(tempDF.at[row,'Element_Annotation'])
                
                if tempDF.at[row,'Tail_Type'] == 'Possible_A-Tail*_and_Possible_T-Tail' or tempDF.at[row,'Tail_Type'] == 'Possible_A-Tail':
                    Tailsequence = str(Seq(tempDF.at[row,'Sequence']).reverse_complement())
                    sequence = str(tempDF.at[row,'Sequence'])
                else:
                    Tailsequence = str(tempDF.at[row,'Sequence'])
                    sequence = str(Seq(tempDF.at[row,'Sequence']).reverse_complement())
                    
                flag=0
                for candidate in ElementDict[element]:
                    if ElementDict[element][candidate]['Seq'] == sequence and flag==0:
                        ElementDict[element][candidate]['Group'].append(name)
                        flag+=1
                    elif ElementDict[element][candidate]['Seq'] == Tailsequence and flag==0:
                        ElementDict[element][candidate]['Group'].append(name)
                        flag+=1
                    else:
                        continue

                if flag==0:
                     ElementDict[element][name]={'Seq':Tailsequence, 'Group':[]}


KeyboardInterrupt: 

In [5]:
import json
#with open("/LeeLab/HPRC/chromosomeY/Data/MEIs/allElement_100Match_MEIs.09152025.json", "w") as f:
#    json.dump(ElementDict, f, indent=4)

with open("/LeeLab/HPRC/chromosomeY/Data/MEIs/allElement_100Match_MEIs.09152025.json", "r") as f:
    loaded_dict = json.load(f)

In [6]:
for element in loaded_dict:
    with open('/LeeLab/HPRC/chromosomeY/Data/MEIs/part3_elements/'+str(element.replace("/","_"))+'_youngElements.09152025.fasta', 'a+') as file:
            for candidate in loaded_dict[element]:
                name = candidate
                sequence = loaded_dict[element][candidate]['Seq']
                file.write(">"+str(name)+"\n")
                file.write(sequence+"\n")
    file.close()

In [47]:
#directory="/LeeLab/HPRC/chromosomeY/Data/MEIs/part2_limeaid/limeaid_MEI_out"
#for file in os.listdir(directory):
#    if '.tsv' in file:
#        limeaid = pd.read_csv(directory+"/"+file,sep='\t')
        #print(len(limeaid))
#        goodElements=['Retroposon/SVA','SINE/Alu','LINE/L1']
#        nonElementDF = limeaid[~limeaid['Element_Designation'].isin(goodElements)].copy()
#        ElementDF = limeaid[limeaid['Element_Designation'].isin(goodElements)].copy()
#        decentINS = ElementDF[(ElementDF['FLAGS']=='No_Flags') & (ElementDF['Tail_Type']!='No_Tail_Type')].copy()

#        for element in goodElements:
#            tempDF = decentINS[decentINS['Element_Designation']==element].copy()
            #with open('/LeeLab/HPRC/chromosomeY/Data/MEIs/part3_elements/'+str(element.replace("/","_"))+'_youngElements.09152025.fasta', 'a+') as file:
#                for row in tempDF.index:
                    
#                    name = str(tempDF.at[row,'ID'])+"-"+str(tempDF.at[row,'Element_Annotation'])
                    
#                    if tempDF.at[row,'Tail_Type'] == 'Possible_A-Tail*_and_Possible_T-Tail' or tempDF.at[row,'Tail_Type'] == 'Possible_A-Tail':
#                        sequence = str(Seq(tempDF.at[row,'Sequence']).reverse_complement())
#                    else:
#                        sequence = str(tempDF.at[row,'Sequence'])

                    #file.write(">"+str(name)+"\n")
                    #file.write(sequence+"\n")
                    
            #file.close()


In [55]:
mobileElements={x:{} for x in set(goodEDF['Element_Designation'])}
mobileElements
for row in tqdm(goodEDF.index):
    myID = str(row)
    sequence = str(goodEDF.at[row,'Sequence'])
    designation = str(goodEDF.at[row,'Element_Designation'])
    myannotation = str(goodEDF.at[row,'Element_Annotation'])
    divergence = float(goodEDF.at[row,'Element_Divergence'])

    if len(mobileElements[designation])==0:
        mobileElements[designation][1] = {'Representative':sequence, 'IDs':[myID], 'Annotation':myannotation, 'Div':divergence}
    else:

        myFlag=0
        for group in mobileElements[designation]:
            groupAnnotation = mobileElements[designation][group]['Annotation']
            groupDivergence = mobileElements[designation][group]['Div']
            groupSequence = mobileElements[designation][group]['Representative']
            
            revGroup = str(Seq(groupSequence).reverse_complement())

            if myFlag ==0 and groupAnnotation == myannotation and abs(divergence-groupDivergence)<.5:
                alignments = pairwise2.align.globalxx(sequence, groupSequence)
                revalignments = pairwise2.align.globalxx(sequence, revGroup)
                aligned_seq1, aligned_seq2, score, start, end = alignments[0]
                revaligned_seq1, revaligned_seq2, score, start, end = revalignments[0]

                orientDistance1 = distance.hamming([x for x in aligned_seq1], [y for y in aligned_seq2])
                orientDistance2 = distance.hamming([x for x in revaligned_seq1], [y for y in revaligned_seq2])

                if orientDistance1<=0.05 or orientDistance2<=0.05:
                    mobileElements[designation][group]['IDs'].append(myID)
                    myFlag+=1
                else:
                    continue

            else:
                continue


        if myFlag==0:
            groupNumber = len(mobileElements[designation])+1
            mobileElements[designation][groupNumber] = {'Representative':sequence, 'IDs':[myID], 'Annotation':myannotation, 'Div':divergence}
        else:
            continue

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9218/9218 [1:47:56<00:00,  1.42it/s]


In [56]:
goodEDF['Clustering_Group']='TEMP'
for family in mobileElements.keys():
    if 'Alu' in family:
        familyName = 'Alu'
    elif 'SVA' in family:
        familyName = 'SVA'

    else:
        familyName='L1'
    
    for group in mobileElements[family]:
        for elementID in mobileElements[family][group]['IDs']:
            goodEDF.at[elementID, 'Clustering_Group'] = familyName+":"+str(group)

In [57]:
import json
#with open("/LeeLab/HPRC/chromosomeY/Data/ColorBlock_MEIs/HPRC_MEI_limeaid_v1.3.2.json", "w") as f:
#    json.dump(mobileElements, f)

In [58]:
#goodEDF.to_csv("/LeeLab/HPRC/chromosomeY/Data/ColorBlock_MEIs/HPRC_v2_limeaidv1.3.2_filtered_wClusters.tsv",sep='\t')